Importy

In [14]:
from plotly import subplots
import pandas as pd
import plotly.graph_objects as go

Przygotowanie danych

In [15]:
cak = pd.read_csv('kgh_d.csv')
cm = pd.read_csv('ca_c_f_d.csv')
cak["Data"] = pd.to_datetime(cak["Data"])
cak["Data"] = cak["Data"].dt.strftime("%Y-%m-%d")
cm["Data"] = pd.to_datetime(cm["Data"])
cm["Data"] = cm["Data"].dt.strftime("%Y-%m-%d")

cak_table = cak[["Data", "Zamkniecie"]].rename(
    columns={"Zamkniecie": "KGH"}
)
cm_table = cm[["Data", "Zamkniecie"]].rename(
    columns={"Zamkniecie": "Miedź"}
)
table_df = cak_table.merge(
    cm_table,
    on="Data",
    how="inner"
)
# table_df["Data"] = table_df["Data"].dt.strftime("%Y-%m-%d")

Wykresy i tabela

Notebook wykorzystuje bibliotekę Plotly. Interaktywne wykresy mogą nie być wyświetlane w podglądzie GitHub i najlepiej otworzyć notebook lokalnie w Jupyter Notebook lub VS Code.

In [16]:
fig = subplots.make_subplots(
    rows=3,
    cols=1,
    subplot_titles=['KGHM','Miedź','Price Table'],
    shared_xaxes=True,
    specs=[
        [{"type": "xy"}],
        [{"type": "xy"}],
        [{"type": "table"}]],
    row_heights=[0.25, 0.25, 0.50]
)
fig.update_layout(
    height=1000,
    margin=dict(l=60,r=60,t=60,b=60)
)
fig.add_trace(go.Scatter(x=cak['Data'],y=cak['Zamkniecie'],name='Cena akcji KGHM zamkniecie'),1,1)
fig.add_trace(go.Scatter(x=cm['Data'],y=cm['Zamkniecie'],name='Cena miedzi zamkniecie'),2,1)
fig.add_trace(go.Table(header=dict(values=table_df.columns,height=28),cells=dict(values=table_df.T.values,height=24)),3,1)


In [17]:
correlation = table_df["KGH"].corr(table_df["Miedź"])
correlation

0.7146244385182003

In [18]:
returns = table_df[["KGH", "Miedź"]].pct_change()
returns.corr()

,KGH,Miedź
KGH,1.000000,0.511901
Miedź,0.511901,1.000000


Obliczona korelacja wskazuje na dodatnią zależność pomiędzy ceną akcji KGHM a ceną miedzi. Po przejściu na dzienne zmiany cen (pct_change) współczynnik korelacji jest niższy, co oznacza, że krótkoterminowe ruchy obu instrumentów są słabiej powiązane niż ich długoterminowe trendy.

In [19]:
returns["Miedź"].corr(returns["KGH"])

0.5119011965417539

In [20]:
returns["Miedź"].corr(returns["KGH"].shift(-1))

-0.0045595909878073165

In [21]:
returns["Miedź"].shift(-1).corr(returns["KGH"])

0.0757050629132741

Tutaj chodziło o odwrotną korelacje KGH do Miedź i Miedź do KGH z różnicą jednej sesji

In [22]:
returns["KGH"].corr(returns["Miedź"].shift(-1))

0.0757050629132741

Obliczona korelacja wskazuje na dodatnią zależność pomiędzy ceną akcji KGHM a ceną miedzi. Po przejściu na dzienne zmiany cen (pct_change) współczynnik korelacji jest niższy, co oznacza, że krótkoterminowe ruchy obu instrumentów są słabiej powiązane niż ich długoterminowe trendy. Sam współczynnik korelacji nie pozwala jednak wnioskować o zależności przyczynowo-skutkowej ani przewidywać przyszłych zmian kursu akcji na podstawie zmian ceny miedzi.

Korelację z przesunięciem o 1 dzień ≈ -0,0046
Zmiana ceny miedzi dzień wcześniej praktycznie nic nie mówi o zmianie ceny KGHM następnego dnia.

In [23]:
corrs = []
for lag in range(0, 31):
    corr = returns["Miedź"].shift(lag).corr(returns["KGH"])
    corrs.append(corr)
fig_corr = go.Figure()
fig_corr.add_trace(
    go.Scatter(x=list(range(31)),y=corrs,mode="lines+markers",name="Korelacja"
    )
)
fig_corr.update_layout(title="Korelacja zmian cen KGHM i miedzi dla różnych opóźnień",
    xaxis_title="Opóźnienie [sesje]",
    yaxis_title="Współczynnik korelacji"
)
fig_corr.show()

In [24]:
corrs = []
for lag in range(31):
    copper_avg = returns["Miedź"].shift(lag).rolling(5).mean()
    corr = copper_avg.corr(returns["KGH"])
    corrs.append(corr)
fig_corr = go.Figure()
fig_corr.add_trace(
    go.Scatter(x=list(range(31)),y=corrs,mode="lines+markers",name="Korelacja"
    )
)
fig_corr.update_layout(title="Korelacja zmian cen KGHM i miedzi dla różnych opóźnień z zastosowaniem okna średniej pięciu sesji",
    xaxis_title="Opóźnienie [sesje]",
    yaxis_title="Współczynnik korelacji"
)
fig_corr.show()

Zastosowanie 5-sesyjnej średniej kroczącej zmniejszyło wpływ przypadkowych jednodniowych zmian, dzięki czemu wykres korelacji jest bardziej stabilny. Nadal jednak poza zerowym opóźnieniem nie występują wyraźne wartości współczynnika korelacji, co nie wskazuje na istnienie prostego opóźnionego związku pomiędzy zmianami ceny miedzi a zmianami kursu akcji KGHM. Przynajmniej dla tych metod analizy.